# Week 2 — Theory
## Manifold learning and high-dimensional embeddings

Neural networks produce representations that live in spaces of hundreds or thousands of
dimensions. We cannot look at $\mathbb{R}^{768}$. The job of a dimensionality reduction
method is to give us a 2D or 3D picture that we *can* look at, while preserving as much
of the structure of the original space as possible.

This notebook covers:

1. The **manifold hypothesis** and why it makes nonlinear methods more appropriate than
   linear ones for representation spaces.
2. **PCA** as a baseline — what it preserves, what it does not.
3. **t-SNE** — its loss function, what `perplexity` controls, and what the resulting
   map can and cannot tell you.
4. **UMAP** — what makes it different from t-SNE, the role of `n_neighbors` and
   `min_dist`, and when one is preferable to the other.
5. **What projections do not preserve** — distances, density, cluster sizes — and how
   to communicate this honestly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

plt.style.use("../../assets/mplstyle/course.mplstyle")
RNG = np.random.default_rng(7)


## 1. The manifold hypothesis

The **manifold hypothesis** states that high-dimensional data of interest — natural
images, sentences, gene expressions — tends to concentrate near a low-dimensional
manifold embedded in the ambient space. Translated:

> The intrinsic dimension of meaningful data is much lower than the dimension of the
> coordinate system we use to store it.

A 224×224 colour image lives in $\mathbb{R}^{150{,}528}$, but almost every point in that
space is white noise. The set of "images of cats" forms an extremely thin sub-region.

A linear method like PCA can only find a linear subspace. If the manifold is curved (and
it almost always is for neural-network representations), PCA either misses the structure
or smears it out.

A small classic example: the **Swiss roll**. Points sampled on a 2D sheet that is rolled
up in 3D. PCA's best 2D approximation collapses the roll into a flat slab, losing the
information that points on opposite layers are actually far apart along the manifold.


In [ ]:
# Swiss roll
n = 1500
t = 1.5 * np.pi * (1 + 2 * RNG.uniform(0, 1, n))
h = 21 * RNG.uniform(0, 1, n)
X = np.stack([t * np.cos(t), h, t * np.sin(t)], axis=1)
X += RNG.normal(0, 0.4, X.shape)

fig = plt.figure(figsize=(13, 4))
ax1 = fig.add_subplot(1, 3, 1, projection="3d")
ax1.scatter(*X.T, c=t, cmap="viridis", s=6)
ax1.set_title("Original 3D Swiss roll")
ax1.view_init(elev=12, azim=-72)

# PCA
pca = PCA(n_components=2).fit_transform(X)
ax2 = fig.add_subplot(1, 3, 2)
ax2.scatter(*pca.T, c=t, cmap="viridis", s=6)
ax2.set_title("PCA — projects 'through' the roll")
ax2.set_aspect("equal")

# UMAP
emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(X)
ax3 = fig.add_subplot(1, 3, 3)
ax3.scatter(*emb.T, c=t, cmap="viridis", s=6)
ax3.set_title("UMAP — unrolls the manifold")
ax3.set_aspect("equal")

plt.tight_layout()
plt.show()


PCA squashes the roll along a fixed axis and loses the ordering along the curve.
UMAP recovers something close to the 2D sheet that the points were originally sampled
from — the colour gradient along $t$ remains monotone across the projection.

This is the qualitative claim "manifold learning works". The next question is *how*.


## 2. PCA — the baseline

PCA finds an orthonormal basis $\{u_1, u_2, \ldots\}$ such that each successive axis
maximizes variance. It is a **linear** method (the projection is a matrix multiplication),
and the optimum has a closed form (eigendecomposition of the data covariance).

What PCA is good at:

- Fast, deterministic, no hyperparameters worth tuning beyond `n_components`.
- Preserves **large-scale geometry** in the directions of highest variance.
- Gives an interpretable **explained-variance ratio** that you can use as a budget.

What PCA misses:

- Any structure that lies along curved directions.
- Cluster separation when clusters are *near* in the high-variance axes but *far* in
  low-variance ones.


In [ ]:
# A toy example where PCA hides a real cluster structure
def two_curves(n=400):
    t = np.linspace(0, 2 * np.pi, n)
    a = np.stack([np.cos(t), np.sin(t), 0.1 * RNG.normal(size=n)], axis=1)
    b = np.stack([np.cos(t), np.sin(t), 5 + 0.1 * RNG.normal(size=n)], axis=1)
    X = np.concatenate([a, b])
    y = np.array([0] * n + [1] * n)
    return X, y

X, y = two_curves()
pca = PCA(n_components=2).fit_transform(X)
emb = umap.UMAP(random_state=0).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(*pca.T, c=y, cmap="coolwarm", s=8); axes[0].set_title("PCA (xy plane)")
axes[1].scatter(*emb.T, c=y, cmap="coolwarm", s=8); axes[1].set_title("UMAP")
for ax in axes: ax.set_aspect("equal")
plt.tight_layout(); plt.show()


Both classes occupy the same $(x, y)$ ring; they differ only in $z$. PCA's first
two components capture the ring (high variance in $x, y$) and discard $z$, hiding the
class structure entirely. A nonlinear method has no such blind spot.


## 3. t-SNE

**t-SNE** (van der Maaten & Hinton, 2008) optimizes a 2D embedding so that
*neighbourhood probabilities* match between the original space and the projection.

In the high-dimensional space, for each point $i$ it defines a Gaussian over neighbours
with a per-point bandwidth $\sigma_i$ chosen so that the **perplexity** is fixed:

$$
P_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \ne i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}
$$

In the 2D space it defines a heavy-tailed (Student's $t$) distribution:

$$
Q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \ne l}(1 + \|y_k - y_l\|^2)^{-1}}
$$

and minimizes the KL divergence $\sum_{ij} P_{ij} \log(P_{ij}/Q_{ij})$.

Two consequences worth committing to memory:

**(a)** t-SNE preserves **local** neighbourhoods, not global geometry. The distance
between two well-separated clusters in the t-SNE plot tells you *almost nothing* about
their distance in the original space.

**(b)** `perplexity` is roughly "how many neighbours each point pays attention to".
Small perplexity → many small clusters. Large perplexity → coarse, sometimes single,
blob. The output is *qualitatively* different at different perplexities; you should
always inspect at least three.


In [ ]:
# Perplexity sweep on a 5-Gaussian mixture
def gaussian_mixture(n_per=200, dim=30, n_clusters=5, sep=4.0):
    centres = RNG.normal(0, sep, size=(n_clusters, dim))
    X = np.concatenate([
        centres[i] + RNG.normal(0, 1, size=(n_per, dim))
        for i in range(n_clusters)])
    y = np.repeat(np.arange(n_clusters), n_per)
    return X, y

X, y = gaussian_mixture()

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for ax, p in zip(axes, [5, 30, 100, 300]):
    emb = TSNE(perplexity=p, random_state=0, init="pca",
               learning_rate="auto").fit_transform(X)
    ax.scatter(*emb.T, c=y, cmap="tab10", s=5)
    ax.set_title(f"perplexity = {p}")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


At `perplexity = 5` t-SNE shatters each Gaussian into sub-clusters that are not
real. At `perplexity = 300` it merges all five into a blob. The "right" answer is
somewhere in the middle — but there is *no* universal right answer; it depends on what
you want the chart to communicate. Default to 30 and check sensitivity.


## 4. UMAP

**UMAP** (McInnes et al., 2018) is based on a different idea — building a fuzzy
simplicial complex of nearest neighbours and finding a low-dimensional layout whose
fuzzy complex matches it. The result is empirically:

- Faster than t-SNE on large datasets.
- Often gives **better global structure** — distances between well-separated clusters
  are more meaningful (though still not metric).
- Two hyperparameters that matter:

  - `n_neighbors`: how local vs. global the embedding is. Small → focus on local
    structure (5–15), large → preserve broader structure (50–200).
  - `min_dist`: how tightly points cluster in the output. Small (0.0–0.1) gives tight
    clumps that are easy to count; large (0.5+) gives diffuse, easier-to-read overlays.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for i, nn in enumerate([5, 30, 200]):
    for j, md_ in enumerate([0.01, 0.3]):
        ax = axes[j, i]
        emb = umap.UMAP(n_neighbors=nn, min_dist=md_,
                        random_state=0).fit_transform(X)
        ax.scatter(*emb.T, c=y, cmap="tab10", s=5)
        ax.set_title(f"n_neighbors={nn}, min_dist={md_}")
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


The 5-cluster Gaussian dataset is easy enough that UMAP recovers the structure
across the grid. On real, noisy data the differences are much larger and the grid is
essential.

### t-SNE vs. UMAP — practical guidance

| If you want…                                  | Reach for… |
|------------------------------------------------|------------|
| A quick chart of a dataset under 5,000 points | Either; t-SNE is fine |
| To embed 100k+ points                         | UMAP, by a wide margin |
| Inter-cluster distances that mean something  | UMAP (with caveats) |
| To **add new points** to an existing layout  | UMAP — supports `.transform()` cleanly |
| To debug a known clustering                  | t-SNE — sharper local separation |

Both should be inspected with at least two hyperparameter settings before you publish.


## 5. What projections do not preserve

A 2D scatter is a thin, lossy summary of the original space. Three honest disclaimers
to keep in mind, and ideally in your figure captions:

1. **Distances are not metric.** A point that looks 5 units from another in the
   embedding may be much closer or much farther in the original space.
2. **Cluster sizes are not informative.** Both t-SNE and UMAP normalize density —
   a tight cluster of 100 points and a tight cluster of 10,000 will look similar.
3. **Empty space is not informative.** A wide gap between two clusters does not imply
   the clusters are "very different"; it may be a layout artefact.

Two diagnostics partly mitigate this:

- **Trustworthiness** (Venna & Kaski, 2001). Quantifies how often points that are
  neighbours in the embedding are also neighbours in the original space.
- **Continuity**. The reverse: how often original-space neighbours remain neighbours
  in the embedding.

Both range in $[0, 1]$. Reporting both numbers in your caption is a cheap way to
demonstrate that you know what your projection is and is not telling the reader.


In [ ]:
from sklearn.manifold import trust_worthiness as trustworthiness
# (sklearn has 'trustworthiness'; the alias above works in older versions too)
from sklearn.manifold import trustworthiness

X, y = gaussian_mixture()
emb_tsne = TSNE(perplexity=30, random_state=0, init="pca",
                learning_rate="auto").fit_transform(X)
emb_umap = umap.UMAP(random_state=0).fit_transform(X)

print(f"t-SNE trustworthiness (k=10): {trustworthiness(X, emb_tsne, n_neighbors=10):.3f}")
print(f"UMAP  trustworthiness (k=10): {trustworthiness(X, emb_umap, n_neighbors=10):.3f}")


## Summary

- The **manifold hypothesis** justifies nonlinear methods; PCA is a useful baseline
  and a sanity check, not the answer.
- Both **t-SNE** and **UMAP** are good defaults. Pick UMAP for scale and for
  out-of-sample transforms; pick t-SNE for local sharpness.
- Inspect **multiple hyperparameter settings**. A single embedding is one hypothesis;
  a grid is evidence.
- Be explicit about what projections **do not** preserve. Use trustworthiness /
  continuity as cheap diagnostics.

In the lab, we apply this to a real text-embedding space and a real image-embedding
space, and we build interactive 3D Plotly views of both.
